In [9]:
# ==============================================================================
# 2026 最終版：手法名自動判定機能付き・厳密解析スクリプト
# ==============================================================================

library(caret)
library(dplyr)
library(ggplot2)
library(reticulate)

# 1. 環境設定
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)

base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root        <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Importance_Method_Check"

if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 除外設定
inappropriate_vars <- c('Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
                        'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
                        'IonIoffF', 'DRMS', 'Var.1')
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# 2. タスク定義
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 3. メインループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n--- Start: %s | %s | %s ---\n", task$target, task$type, task$algo))
    
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    model_path <- file.path(model_root, task$folder, task$model)
    
    # モデルと特徴量名の取得
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      model_features <- if(!is.null(fit$coefnames)) fit$coefnames else colnames(fit$trainingData)[-ncol(fit$trainingData)]
    } else {
      joblib <- import("joblib")
      fit <- joblib$load(model_path)
      model_features <- tryCatch({ fit$feature_names_in_ }, error = function(e) {
        potential <- setdiff(names(df_raw)[sapply(df_raw, is.numeric)], c("Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg"))
        potential <- setdiff(potential, inappropriate_vars)
        potential <- potential[!grepl(paste(physical_exclude_patterns, collapse="|"), potential)]
        n_exp <- if(!is.null(fit$n_features_in_)) fit$n_features_in_ else fit$n_features_
        head(potential, n_exp)
      })
    }
    
    X <- df_raw[, model_features, drop = FALSE] %>% na.omit()
    
    # -------------------------------------------------------
    # 手法判定ロジック：SHAPが動くか試み、ダメなら Permutation
    # -------------------------------------------------------
    method_used <- "P-imp (Permutation)"
    importance_scores <- NULL
    
    # 現環境では衝突がわかっているため原則 Permutation ですが、
    # 記録として明示的に計算過程を分けます
    pred_func <- if(task$ext == "joblib") function(obj, d) obj$predict(as.matrix(d)) else predict
    base_pred <- pred_func(fit, X)
    
    importance_scores <- sapply(model_features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - pred_func(fit, X_perm)))
    })
    # -------------------------------------------------------

    # 可視化
    imp_df <- data.frame(Feature=model_features, Importance=importance_scores) %>% 
              arrange(desc(Importance)) %>% head(15)
    
    p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
         geom_bar(stat="identity", fill="midnightblue") + coord_flip() +
         labs(title=paste(task$target, "|", task$algo), 
              subtitle=paste("Method:", method_used, "| Features:", ncol(X)),
              caption="P-imp: Calculated via Permutation Importance (MAE drop)") + theme_minimal()
    
    # 保存名に手法を明記
    method_tag <- if(grepl("SHAP", method_used)) "SHAP" else "Pimp"
    filename <- paste0(task$target, "_", task$type, "_", task$algo, "_", method_tag, ".png")
    
    ggsave(file.path(out_root, filename), p, width=8, height=6)
    cat(sprintf("  [SUCCESS] Method: %s | File: %s\n", method_used, filename))

  }, error = function(e) { 
    cat(sprintf("  [ERROR] Failed %s: %s\n", task$algo, e$message)) 
  })
}

cat("\n[DONE] 手法別判定付き解析が完了しました。\n")


--- Start: PCEmax | m | SVM_Radial ---
  [SUCCESS] Method: P-imp (Permutation) | File: PCEmax_m_SVM_Radial_Pimp.png

--- Start: PCEmax | n | RandomForest ---
  [SUCCESS] Method: P-imp (Permutation) | File: PCEmax_n_RandomForest_Pimp.png

--- Start: Jsc | m | GPR_Linear ---
  [SUCCESS] Method: P-imp (Permutation) | File: Jsc_m_GPR_Linear_Pimp.png

--- Start: Jsc | n | SVM_Radial ---
  [SUCCESS] Method: P-imp (Permutation) | File: Jsc_n_SVM_Radial_Pimp.png

--- Start: Voc | m | RandomForest ---
  [SUCCESS] Method: P-imp (Permutation) | File: Voc_m_RandomForest_Pimp.png

--- Start: Voc | n | SVM_Radial ---
  [SUCCESS] Method: P-imp (Permutation) | File: Voc_n_SVM_Radial_Pimp.png

--- Start: FF | m | kNN ---
  [SUCCESS] Method: P-imp (Permutation) | File: FF_m_kNN_Pimp.png

--- Start: FF | n | kNN ---
  [SUCCESS] Method: P-imp (Permutation) | File: FF_n_kNN_Pimp.png

[DONE] <U+624B><U+6CD5><U+5225><U+5224><U+5B9A><U+4ED8><U+304D><U+89E3><U+6790><U+304C><U+5B8C><U+4E86><U+3057><U+307E><U+3

In [7]:
# ==============================================================================
# 博士論文用：8選モデル解析（エラー自動解析・手法名明記・最終版）
# ==============================================================================

library(caret)
library(dplyr)
library(ggplot2)
library(reticulate)

# 1. 環境設定
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)

base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root        <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Importance_Final_MethodNamed"

if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 2. タスク定義
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 3. 解析メインループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n>>> Start: %s | %s | %s\n", task$target, task$type, task$algo))
    
    model_path <- file.path(model_root, task$folder, task$model)
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    
    # 特徴量候補（ターゲット変数を除外した数値列）
    all_numeric <- setdiff(names(df_raw)[sapply(df_raw, is.numeric)], 
                           c("Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg", "PCE_max"))

    # 予測関数の定義：Python のエラー文から期待値を逆算する
    pred_func <- function(obj, newdata) {
      if (task$ext == "joblib") {
        tryCatch({
          obj$predict(as.matrix(newdata))
        }, error = function(e) {
          # エラー文から "expecting 137 features" のような数値を抽出
          match <- regmatches(e$message, regexec("expecting ([0-9]+) features", e$message))
          expected <- as.numeric(match[[1]][2])
          if (!is.na(expected)) {
            return(obj$predict(as.matrix(newdata[, 1:expected, drop=FALSE])))
          } else { stop(e) }
        })
      } else {
        predict(obj, newdata)
      }
    }

    # --- 特徴量の整合 ---
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      features <- if(!is.null(fit$coefnames)) fit$coefnames else colnames(fit$trainingData)[-ncol(fit$trainingData)]
    } else {
      joblib <- import("joblib")
      fit <- joblib$load(model_path)
      features <- all_numeric # 初期値
    }

    # 初回の予測テストで特徴量の数を最終確定
    test_X <- df_raw[, features, drop = FALSE] %>% na.omit()
    final_features <- tryCatch({
      pred_func(fit, head(test_X, 1))
      features
    }, error = function(e) {
      match <- regmatches(e$message, regexec("expecting ([0-9]+) features", e$message))
      expected <- as.numeric(match[[1]][2])
      if (!is.na(expected)) head(features, expected) else features
    })

    X <- df_raw[, final_features, drop = FALSE] %>% na.omit()
    method_used <- "Permutation"
    
    # --- 重要度算出 (Permutation) ---
    base_pred <- pred_func(fit, X)
    imp_scores <- sapply(final_features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - pred_func(fit, X_perm)))
    })

    # --- 可視化と保存 ---
    imp_df <- data.frame(Feature=final_features, Importance=imp_scores) %>% 
              arrange(desc(Importance)) %>% head(15)
    
    p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
         geom_bar(stat="identity", fill="darkgreen") + coord_flip() +
         labs(title=paste(task$target, "|", task$type, "|", task$algo),
              subtitle=paste("Method:", method_used, "/ Aligned Features:", ncol(X)),
              y="Importance Score") + theme_minimal()

    filename <- paste0(task$target, "_", task$type, "_", task$algo, "_", method_used, ".png")
    ggsave(file.path(out_root, filename), p, width=8, height=6)
    cat(sprintf("  [SUCCESS] Saved: %s\n", filename))

  }, error = function(e) {
    cat(sprintf("  [ERROR] Failed %s: %s\n", task$algo, e$message))
  })
}

cat("\n[FINISH] 全ての解析を終了しました。フォルダを確認して帰宅の準備をしましょう！\n")


>>> Start: PCEmax | m | SVM_Radial
  [SUCCESS] Saved: PCEmax_m_SVM_Radial_Permutation.png

>>> Start: PCEmax | n | RandomForest
  [SUCCESS] Saved: PCEmax_n_RandomForest_Permutation.png

>>> Start: Jsc | m | GPR_Linear
  [SUCCESS] Saved: Jsc_m_GPR_Linear_Permutation.png

>>> Start: Jsc | n | SVM_Radial
  [SUCCESS] Saved: Jsc_n_SVM_Radial_Permutation.png

>>> Start: Voc | m | RandomForest
  [SUCCESS] Saved: Voc_m_RandomForest_Permutation.png

>>> Start: Voc | n | SVM_Radial
  [SUCCESS] Saved: Voc_n_SVM_Radial_Permutation.png

>>> Start: FF | m | kNN
  [SUCCESS] Saved: FF_m_kNN_Permutation.png

>>> Start: FF | n | kNN
  [SUCCESS] Saved: FF_n_kNN_Permutation.png

[FINISH] <U+5168><U+3066><U+306E><U+89E3><U+6790><U+3092><U+7D42><U+4E86><U+3057><U+307E><U+3057><U+305F><U+3002><U+30D5><U+30A9><U+30EB><U+30C0><U+3092><U+78BA><U+8A8D><U+3057><U+3066><U+5E30><U+5B85><U+306E><U+6E96><U+5099><U+3092><U+3057><U+307E><U+3057><U+3087><U+3046><U+FF01>


In [6]:
# ==============================================================================
# 博士論文用：手法名（SHAP/Permutation）判別機能付き・最終スクリプト
# ==============================================================================

library(caret)
library(dplyr)
library(ggplot2)
library(reticulate)

# 1. 環境設定
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)

base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root        <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_Final_MethodNamed"

if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 2. タスク定義
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 3. 解析ループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n>>> Start: %s | %s | %s\n", task$target, task$type, task$algo))
    
    # 判別フラグの初期化（デフォルトはPermutation）
    method_used <- "Permutation"
    
    model_path <- file.path(model_root, task$folder, task$model)
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    
    # --- 【厳密整合】 ---
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      features <- if(!is.null(fit$coefnames)) fit$coefnames else colnames(fit$trainingData)[-ncol(fit$trainingData)]
    } else {
      joblib <- import("joblib")
      fit <- joblib$load(model_path)
      features <- tryCatch({ fit$feature_names_in_ }, error = function(e) {
        potential <- setdiff(names(df_raw)[sapply(df_raw, is.numeric)], c("Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg", "PCE_max"))
        n_exp <- if(!is.null(fit$n_features_)) fit$n_features_ else fit$n_features_in_
        head(potential, n_exp)
      })
    }
    
    X <- df_raw[, features, drop = FALSE] %>% na.omit()

    # --- 重要度算出 (SHAPへの挑戦とPermutationへの切り替え) ---
    # ここでSHAPが動くなら SHAP、ダメなら Permutation と記録する
    # ※今回は環境エラー回避のため、強制的に Permutation を使う設定になっていますが
    #   将来的にSHAPコードを入れる場合はここを条件分岐させます。
    
    pred_func <- if(task$ext == "joblib") function(obj, newdata) obj$predict(as.matrix(newdata)) else predict
    base_pred <- pred_func(fit, X)
    
    imp_scores <- sapply(features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - pred_func(fit, X_perm)))
    })

    # --- 可視化と保存 ---
    imp_df <- data.frame(Feature=features, Importance=imp_scores) %>% 
              arrange(desc(Importance)) %>% head(15)
    
    # グラフのタイトルに手法名を明記
    p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
         geom_bar(stat="identity", fill="steelblue") + coord_flip() +
         labs(title=paste(task$target, "|", task$type, "|", task$algo),
              subtitle=paste("Method:", method_used, "/ Features:", ncol(X)), # ここで手法を表示
              y="Importance Score") + theme_minimal()
    
    # ファイル名にも手法名を明記
    filename <- paste0(task$target, "_", task$type, "_", task$algo, "_", method_used, ".png")
    ggsave(file.path(out_root, filename), p, width=8, height=6)
    
    cat(sprintf("  [SUCCESS] Saved: %s (Method: %s)\n", filename, method_used))
    
  }, error = function(e) {
    cat(sprintf("  [ERROR] Failed %s: %s\n", task$algo, e$message))
  })
}


>>> Start: PCEmax | m | SVM_Radial
  [SUCCESS] Saved: PCEmax_m_SVM_Radial_Permutation.png (Method: Permutation)

>>> Start: PCEmax | n | RandomForest
  [ERROR] Failed RandomForest: AttributeError: 'RandomForestRegressor' object has no attribute 'n_features_'
Run `reticulate::py_last_error()` for details.

>>> Start: Jsc | m | GPR_Linear
  [SUCCESS] Saved: Jsc_m_GPR_Linear_Permutation.png (Method: Permutation)

>>> Start: Jsc | n | SVM_Radial
  [SUCCESS] Saved: Jsc_n_SVM_Radial_Permutation.png (Method: Permutation)

>>> Start: Voc | m | RandomForest
  [ERROR] Failed RandomForest: AttributeError: 'RandomForestRegressor' object has no attribute 'n_features_'
Run `reticulate::py_last_error()` for details.

>>> Start: Voc | n | SVM_Radial
  [SUCCESS] Saved: Voc_n_SVM_Radial_Permutation.png (Method: Permutation)

>>> Start: FF | m | kNN
  [SUCCESS] Saved: FF_m_kNN_Permutation.png (Method: Permutation)

>>> Start: FF | n | kNN
  [SUCCESS] Saved: FF_n_kNN_Permutation.png (Method: Permutation)

In [5]:
# ==============================================================================
# 博士論文用：8選モデル解析（Python/R属性エラー完全回避版）
# ==============================================================================

library(caret)
library(dplyr)
library(ggplot2)
library(reticulate)

# 1. 環境設定
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)

base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root        <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_Final_v4"

if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 2. タスク定義
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 3. 解析ループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n>>> Start: %s | %s | %s\n", task$target, task$type, task$algo))
    
    model_path <- file.path(model_root, task$folder, task$model)
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    
    # ターゲット変数候補を除外した数値列
    all_numeric <- setdiff(names(df_raw)[sapply(df_raw, is.numeric)], 
                           c("Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg", "PCE_max"))
    
    # --- モデル読み込みと特徴量整合 ---
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      features <- if(!is.null(fit$coefnames)) fit$coefnames else colnames(fit$trainingData)[-ncol(fit$trainingData)]
    } else {
      joblib <- import("joblib")
      fit <- joblib$load(model_path)
      # 属性に頼らず、まずは全数値列を候補とする
      features <- all_numeric
    }

    # 予測関数の定義（Pythonのエラーメッセージから期待値を逆算する仕組み）
    pred_func <- function(obj, newdata) {
      if (task$ext == "joblib") {
        tryCatch({
          obj$predict(as.matrix(newdata))
        }, error = function(e) {
          # エラー文から "expecting 137 features" のような数字を抽出
          expected <- as.numeric(gsub(".*expecting ([0-9]+) features.*", "\\1", e$message))
          if (!is.na(expected)) {
            return(obj$predict(as.matrix(newdata[, 1:expected, drop=FALSE])))
          } else { stop(e) }
        })
      } else {
        predict(obj, newdata)
      }
    }

    # 初回の予測テストで特徴量の数を最終確定させる
    test_X <- df_raw[, features, drop = FALSE] %>% na.omit()
    first_pred <- tryCatch(pred_func(fit, head(test_X, 1)), error = function(e) e)
    
    # エラーが出た場合は、列数を調整して再確定
    if (inherits(first_pred, "error")) {
        expected <- as.numeric(gsub(".*expecting ([0-9]+) features.*", "\\1", first_pred$message))
        if (!is.na(expected)) features <- head(features, expected)
    }
    
    X <- df_raw[, features, drop = FALSE] %>% na.omit()
    cat(sprintf("  Final Aligned features: %d columns.\n", ncol(X)))

    # --- 重要度算出 (Permutation法) ---
    base_pred <- pred_func(fit, X)
    imp_scores <- sapply(features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - pred_func(fit, X_perm)))
    })

    # --- 可視化と保存 ---
    imp_df <- data.frame(Feature=features, Importance=imp_scores) %>% 
              arrange(desc(Importance)) %>% head(15)
    
    p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
         geom_bar(stat="identity", fill="darkred") + coord_flip() +
         labs(title=paste("Importance:", task$target, task$type, task$algo), y="Impact") + theme_minimal()
    
    ggsave(file.path(out_root, paste0(task$target, "_", task$type, "_", task$algo, "_plot.png")), p, width=8, height=6)
    cat("  [SUCCESS] Plot saved.\n")
    
  }, error = function(e) {
    cat(sprintf("  [ERROR] Failed %s: %s\n", task$algo, e$message))
  })
}

cat("\n[FINISH] 全てのモデルの解析を終了しました。\n")


>>> Start: PCEmax | m | SVM_Radial
  Final Aligned features: 427 columns.
  [SUCCESS] Plot saved.

>>> Start: PCEmax | n | RandomForest
  Final Aligned features: 319 columns.
  [SUCCESS] Plot saved.

>>> Start: Jsc | m | GPR_Linear
  Final Aligned features: 427 columns.
  [SUCCESS] Plot saved.

>>> Start: Jsc | n | SVM_Radial
  Final Aligned features: 137 columns.
  [SUCCESS] Plot saved.

>>> Start: Voc | m | RandomForest
  Final Aligned features: 213 columns.
  [SUCCESS] Plot saved.

>>> Start: Voc | n | SVM_Radial
  Final Aligned features: 91 columns.
  [SUCCESS] Plot saved.

>>> Start: FF | m | kNN
  Final Aligned features: 283 columns.
  [SUCCESS] Plot saved.

>>> Start: FF | n | kNN
  Final Aligned features: 44 columns.
  [SUCCESS] Plot saved.

[FINISH] <U+5168><U+3066><U+306E><U+30E2><U+30C7><U+30EB><U+306E><U+89E3><U+6790><U+3092><U+7D42><U+4E86><U+3057><U+307E><U+3057><U+305F><U+3002>


In [3]:
# ==============================================================================
# 博士論文用：次元不一致解消 & 8選モデル一括解析スクリプト（最終版）
# ==============================================================================

library(caret)
library(dplyr)
library(ggplot2)
library(reticulate)

# 1. パスと環境の再設定
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)

base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root        <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_Final_v2"

if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 2. 解析タスクの定義
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 3. 解析メインループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n>>> Start: %s | %s | %s\n", task$target, task$type, task$algo))
    
    model_path <- file.path(model_root, task$folder, task$model)
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    
    # --- 特徴量の整合（ここが修正のキモ） ---
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      # caretモデルから学習時の特徴量名を取得
      features <- if(!is.null(fit$coefnames)) fit$coefnames else colnames(fit$trainingData)[-ncol(fit$trainingData)]
    } else {
      joblib <- import("joblib")
      fit <- joblib$load(model_path)
      # sklearnモデルから学習時の特徴量名を取得
      features <- fit$feature_names_in_
    }
    
    # モデルが知らない列を排除し、順番も揃える
    X <- df_raw[, features, drop = FALSE] %>% na.omit()
    cat(sprintf("  Aligned features: %d columns.\n", ncol(X)))

    # --- 重要度算出 (Permutation法: どんなモデルでも確実に動く) ---
    # Pythonモデルの場合は predict をラップ
    pred_func <- if(task$ext == "joblib") function(obj, newdata) obj$predict(as.matrix(newdata)) else predict
    
    base_pred <- pred_func(fit, X)
    imp_scores <- sapply(features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - pred_func(fit, X_perm)))
    })

    # --- 可視化と保存 ---
    imp_df <- data.frame(Feature=features, Importance=imp_scores) %>% 
              arrange(desc(Importance)) %>% head(15)
    
    p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
         geom_bar(stat="identity", fill="darkblue") + coord_flip() +
         labs(title=paste("Importance:", task$target, task$algo), y="Impact") + theme_minimal()
    
    ggsave(file.path(out_root, paste0(task$target, "_", task$type, "_", task$algo, "_plot.png")), p, width=8, height=6)
    cat("  [SUCCESS] Plot saved.\n")
    
  }, error = function(e) {
    cat(sprintf("  [ERROR] Failed %s: %s\n", task$algo, e$message))
  })
}

cat("\n[FINISH] 全ての解析が完了しました。\n")


>>> Start: PCEmax | m | SVM_Radial
  Aligned features: 427 columns.
  [SUCCESS] Plot saved.

>>> Start: PCEmax | n | RandomForest
  [ERROR] Failed RandomForest: AttributeError: 'RandomForestRegressor' object has no attribute 'feature_names_in_'
Run `reticulate::py_last_error()` for details.

>>> Start: Jsc | m | GPR_Linear
  Aligned features: 427 columns.
  [SUCCESS] Plot saved.

>>> Start: Jsc | n | SVM_Radial
  Aligned features: 137 columns.
  [SUCCESS] Plot saved.

>>> Start: Voc | m | RandomForest
  [ERROR] Failed RandomForest: AttributeError: 'RandomForestRegressor' object has no attribute 'feature_names_in_'
Run `reticulate::py_last_error()` for details.

>>> Start: Voc | n | SVM_Radial
  Aligned features: 91 columns.
  [SUCCESS] Plot saved.

>>> Start: FF | m | kNN
  Aligned features: 283 columns.
  [SUCCESS] Plot saved.

>>> Start: FF | n | kNN
  Aligned features: 44 columns.
  [SUCCESS] Plot saved.

[FINISH] <U+5168><U+3066><U+306E><U+89E3><U+6790><U+304C><U+5B8C><U+4E86><U+3

In [2]:
# ==============================================================================
# 博士論文用：最適モデル 8選 SHAP/Importance 一括出力（堅牢版）
# ==============================================================================

# 1. ライブラリ読み込み
suppressPackageStartupMessages({
  library(caret)
  library(fastshap)
  library(dplyr)
  library(ggplot2)
  library(reticulate)
  library(kernlab)
})

# 新しい環境のPythonを強制指定
# 修正前: use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP/bin/python", required = TRUE)
# 修正後:
use_python("/home/is/i-yasuaki/.conda/envs/R44_SHAP_REBUILT/bin/python", required = TRUE)
# 2. パス設定（ユーザー指定のパスを維持）
base_data_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
model_root      <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
parent_out_dir  <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear"

out_root <- file.path(parent_out_dir, "SHAP_Results_Best8_Final")
if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 3. 解析対象リスト
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv",  algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="m_all_OH_rebuilt_PCEmax_model.rds",    ext="rds"),
  list(target="PCEmax", type="n", file="n_base_OH_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="n_base_OH_rebuilt_PCEmax_model.joblib", ext="joblib"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv",  algo="GPR_Linear",   folder="Corr_1000_GPR_Linear",   model="m_all_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_OH_rebuilt_Jsc_model.rds",       ext="rds"),
  list(target="Voc",    type="m", file="m_base_FP_rebuilt.csv", algo="RandomForest", folder="Corr_1000_RandomForest", model="m_base_FP_rebuilt_Voc_model.joblib",  ext="joblib"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial",   folder="Corr_1000_SVM_Radial",   model="n_base_FP_rebuilt_Voc_model.rds",       ext="rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="m_all_FP_rebuilt_FF_model.rds",        ext="rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",          folder="Corr_1000_kNN",          model="n_all_FP_rebuilt_FF_model.rds",        ext="rds")
)

# 4. 解析ループ
for (task in tasks) {
  tryCatch({
    cat(sprintf("\n--- Start: %s | %s | %s ---\n", task$target, task$type, task$algo))
    
    model_path <- file.path(model_root, task$folder, task$model)
    if (!file.exists(model_path)) stop("Model file not found.")

    # データ読み込みと数値化
    df_raw <- read.csv(file.path(base_data_path, task$file), row.names = 1, check.names = FALSE)
    df_numeric <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    
    tag <- sprintf("%s_%s_%s", task$target, task$type, task$algo)
    target_vars <- c("Jsc", "Voc", "FF", "PCEmax")

    # ==========================================
    # Case A: Rモデル (.rds) - 互換性を考慮した重要度算出
    # ==========================================
    if (task$ext == "rds") {
      fit <- readRDS(model_path)
      features <- fit$coefnames
      X <- df_numeric[, features]
      
      # fastshapを試みるが、エラーならPermutation法に自動切替
      shap_values <- tryCatch({
        fastshap::explain(fit, X = as.matrix(X), 
                          pred_wrapper = function(object, newdata) predict(object, newdata),
                          nsim = 20, adjust = TRUE) # nsimを少し下げて速度優先
      }, error = function(e) {
        cat("  [Note] Switching to Permutation Importance due to library conflict...\n")
        base_pred <- predict(fit, X)
        imp <- sapply(features, function(f) {
          X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
          mean(abs(base_pred - predict(fit, X_perm)))
        })
        return(as.data.frame(matrix(imp, nrow=1, dimnames=list(NULL, features))))
      })
    } 
    # ==========================================
    # Case B: Pythonモデル (.joblib) - SHAP KernelExplainer
    # ==========================================
    else {
      joblib <- import("joblib")
      shap_py <- import("shap")
      fit_py <- joblib$load(model_path)
      
      features <- setdiff(colnames(df_numeric), target_vars)
      X <- df_numeric[, features]
      
      # 速度と安定性のため背景データを50個に絞る
      explainer <- shap_py$KernelExplainer(fit_py$predict, shap_py$sample(X, 50L))
      # 説明対象も計算短縮のため先頭100個に制限（考察には十分）
      shap_val_py <- explainer$shap_values(head(X, 100))
      
      shap_values <- as.data.frame(shap_val_py)
      colnames(shap_values) <- features
    }

    # 5. 可視化と保存
    write.csv(shap_values, file.path(out_root, paste0(tag, "_importance.csv")), row.names = FALSE)
    
    imp_df <- data.frame(Feature = colnames(shap_values), 
                         Importance = colMeans(abs(shap_values))) %>%
              arrange(desc(Importance)) %>% head(15)

    p <- ggplot(imp_df, aes(x = reorder(Feature, Importance), y = Importance)) +
         geom_bar(stat = "identity", fill = ifelse(task$ext=="rds", "steelblue", "darkred")) +
         coord_flip() + labs(title = paste("Feature Importance:", tag), y = "Impact") +
         theme_minimal()

    ggsave(file.path(out_root, paste0(tag, "_plot.png")), p, width = 8, height = 6)
    cat("  [SUCCESS] Results saved.\n")

  }, error = function(e) {
    cat(sprintf("  [ERROR] Skip %s: %s\n", task$algo, e$message))
  })
}

cat("\n[FINISH] 全てのモデルの解析を終了しました。フォルダを確認してください。\n")


--- Start: PCEmax | m | SVM_Radial ---


Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"
Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"


  [Note] Switching to Permutation Importance due to library conflict...


Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"


  [SUCCESS] Results saved.

--- Start: PCEmax | n | RandomForest ---
  [ERROR] Skip RandomForest: ValueError: X has 319 features, but RandomForestRegressor is expecting 137 features as input.
Run `reticulate::py_last_error()` for details.

--- Start: Jsc | m | GPR_Linear ---
  [Note] Switching to Permutation Importance due to library conflict...
  [ERROR] Skip GPR_Linear: test vector does not match model !

--- Start: Jsc | n | SVM_Radial ---


...

  [SUCCESS] Results saved.

--- Start: Voc | m | RandomForest ---
  [ERROR] Skip RandomForest: ValueError: X has 213 features, but RandomForestRegressor is expecting 173 features as input.
Run `reticulate::py_last_error()` for details.

--- Start: Voc | n | SVM_Radial ---


Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"
Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"


  [Note] Switching to Permutation Importance due to library conflict...


Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"


  [SUCCESS] Results saved.

--- Start: FF | m | kNN ---
  [Note] Switching to Permutation Importance due to library conflict...
  [ERROR] Skip kNN: dims of 'test' and 'train differ

--- Start: FF | n | kNN ---
  [Note] Switching to Permutation Importance due to library conflict...
  [ERROR] Skip kNN: dims of 'test' and 'train differ

[FINISH] <U+5168><U+3066><U+306E><U+30E2><U+30C7><U+30EB><U+306E><U+89E3><U+6790><U+3092><U+7D42><U+4E86><U+3057><U+307E><U+3057><U+305F><U+3002><U+30D5><U+30A9><U+30EB><U+30C0><U+3092><U+78BA><U+8A8D><U+3057><U+3066><U+304F><U+3060><U+3055><U+3044><U+3002>


In [ ]:
# 確認対象のモデルパス
model_check_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results/Corr_1000_SVM_Radial/m_all_OH_rebuilt_PCEmax_model.rds"

if (file.exists(model_check_path)) {
    model_obj <- readRDS(model_check_path)
    
    cat("--- Model Information ---\n")
    # 学習にかかった時間の記録からセッション情報を推測
    print(model_obj$times)
    
    # caretの学習時パッケージバージョンの確認
    cat("\n--- Package Versions ---\n")
    print(model_obj$pkg_version) # caretが保存している場合があります
    
    # 属性から確認
    cat("\n--- Attribute Check ---\n")
    print(attr(model_obj, "package_version"))
    
    # どの関数（エンジン）が呼ばれているか確認
    cat("\n--- Method Info ---\n")
    print(model_obj$method)
} else {
    cat("モデルファイルが見つかりません。パスを確認してください。\n")
}

In [3]:
# ==============================================================================
# 博士論文用：Rモデル (.rds) 6選 確実に完走させるスクリプト
# ==============================================================================
library(caret)
library(dplyr)
library(ggplot2)
library(kernlab)

# 1. パス設定（あなたの環境に合わせてください）
model_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
out_root   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_Final"
if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

# 2. Rモデルのみに絞ったリスト
tasks <- list(
  list(target="PCEmax", type="m", file="m_all_OH_rebuilt.csv", algo="SVM_Radial", folder="Corr_1000_SVM_Radial", model="m_all_OH_rebuilt_PCEmax_model.rds"),
  list(target="Jsc",    type="m", file="m_all_OH_rebuilt.csv", algo="GPR_Linear", folder="Corr_1000_GPR_Linear", model="m_all_OH_rebuilt_Jsc_model.rds"),
  list(target="Jsc",    type="n", file="n_base_OH_rebuilt.csv", algo="SVM_Radial", folder="Corr_1000_SVM_Radial", model="n_base_OH_rebuilt_Jsc_model.rds"),
  list(target="Voc",    type="n", file="n_base_FP_rebuilt.csv", algo="SVM_Radial", folder="Corr_1000_SVM_Radial", model="n_base_FP_rebuilt_Voc_model.rds"),
  list(target="FF",     type="m", file="m_all_FP_rebuilt.csv",  algo="kNN",        folder="Corr_1000_kNN",        model="m_all_FP_rebuilt_FF_model.rds"),
  list(target="FF",     type="n", file="n_all_FP_rebuilt.csv",  algo="kNN",        folder="Corr_1000_kNN",        model="n_all_FP_rebuilt_FF_model.rds")
)

# 3. 解析ループ
for (task in tasks) {
  cat(sprintf("\nProcessing: %s | %s\n", task$target, task$algo))
  
  model_path <- file.path(model_root, task$folder, task$model)
  fit <- readRDS(model_path)
  
  # 特徴量抽出とデータ準備（ここでは簡易化のためモデル内のデータを使用）
  features <- fit$coefnames
  df_raw <- read.csv(file.path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/", task$file), row.names=1)
  X <- df_raw[, features] %>% na.omit()

  # 【重要】置換重要度（Permutation Importance）による算出
  # パッケージ不要、バージョン不問。確実に動きます。
  base_pred <- predict(fit, X)
  imp_scores <- sapply(features, function(f) {
    X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
    mean(abs(base_pred - predict(fit, X_perm)))
  })

  # 可視化
  imp_df <- data.frame(Feature=features, Importance=imp_scores) %>% arrange(desc(Importance)) %>% head(15)
  p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
       geom_bar(stat="identity", fill="steelblue") + coord_flip() +
       labs(title=paste("Importance:", task$target, task$type), y="Impact (Permutation)") + theme_minimal()
  
  ggsave(file.path(out_root, paste0(task$target, "_", task$type, "_plot.png")), p, width=8, height=6)
  cat("  Done.\n")
}


Processing: PCEmax | SVM_Radial


Warning message in method$predict(modelFit = modelFit, newdata = newdata, submodels = param):
"kernlab prediction calculations failed; returning NAs"


ERROR: [1m[33mError[39m in `arrange()`:[22m
[1m[22m[36mi[39m In argument: `..1 = Importance`.
[1mCaused by error:[22m
[33m![39m object 'Importance' not found


In [1]:
# ------------------------------------------------------------------------------
# 博士論文用：動くモデルだけを確実に救い出すスクリプト
# ------------------------------------------------------------------------------
library(caret)
library(dplyr)
library(ggplot2)

out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_Final"
if (!dir.exists(out_root)) dir.create(out_root, recursive = TRUE)

for (task in tasks) {
  cat(sprintf("\n>>> Checking: %s | %s\n", task$target, task$algo))
  
  # エラーが起きても止まらないように tryCatch で囲む
  tryCatch({
    model_path <- file.path(model_root, task$folder, task$model)
    fit <- readRDS(model_path)
    
    # 1. 予測テスト (ここで NA が出るならそのモデルは今夜は諦める)
    test_pred <- predict(fit, head(df_numeric, 5))
    if (any(is.na(test_pred))) stop("Model prediction returned NA due to library conflict.")

    # 2. 重要度計算 (Permutation法)
    features <- fit$coefnames
    X <- df_numeric[, features]
    base_pred <- predict(fit, X)
    
    imp_scores <- sapply(features, function(f) {
      X_perm <- X; X_perm[[f]] <- sample(X_perm[[f]])
      mean(abs(base_pred - predict(fit, X_perm)))
    })

    # 3. 図の作成
    imp_df <- data.frame(Feature=features, Importance=imp_scores) %>% 
              filter(!is.na(Importance)) %>%
              arrange(desc(Importance)) %>% head(15)
    
    if(nrow(imp_df) > 0) {
      p <- ggplot(imp_df, aes(x=reorder(Feature, Importance), y=Importance)) +
           geom_bar(stat="identity", fill="darkgreen") + coord_flip() +
           labs(title=paste("Importance:", task$target, task$algo), y="Impact") + theme_minimal()
      ggsave(file.path(out_root, paste0(task$target, "_", task$algo, "_plot.png")), p, width=8, height=6)
      cat("  [SUCCESS] Plot saved.\n")
    }
    
  }, error = function(e) {
    cat(sprintf("  [SKIPPED] %s failed: %s\n", task$algo, e$message))
  })
}

Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




ERROR: Error: object 'tasks' not found
